In [1]:
import pandas as pd
import pyodbc

In [2]:
conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 18 for SQL Server};"
    r"SERVER=.\SQLEXPRESS;"
    r"DATABASE=ShopifyMarketplace;"
    r"Trusted_Connection=yes;"
    r"TrustServerCertificate=yes;"
)

print("Connected successfully!")

Connected successfully!


In [3]:
# Phase 8 - Step 3: Load category opportunity metrics into Python
# Phase 8 - 第三步：将 Category 机会评估指标读取到 Python

query = """
WITH category_app_stats AS (
    SELECT
        c.id AS category_id,
        c.title AS category,
        COUNT(DISTINCT ac.app_id) AS app_count,
        COUNT(DISTINCT a.developer) AS developer_count,
        AVG(a.rating) AS avg_rating
    FROM apps_categories ac
    JOIN categories c
        ON ac.category_id = c.id
    JOIN apps a
        ON ac.app_id = a.id
    WHERE a.rating IS NOT NULL
    GROUP BY c.id, c.title
),

category_review_stats AS (
    SELECT
        ac.category_id,
        COUNT(*) AS review_count,
        SUM(CASE WHEN r.rating = 1 THEN 1 ELSE 0 END) AS one_star_reviews,
        SUM(CASE WHEN r.has_developer_reply = 1 THEN 1 ELSE 0 END) AS replied_reviews
    FROM apps_categories ac
    JOIN reviews r
        ON ac.app_id = r.app_id
    GROUP BY ac.category_id
),

app_review_counts AS (
    SELECT
        ac.category_id,
        ac.app_id,
        COUNT(r.rating) AS app_review_count
    FROM apps_categories ac
    LEFT JOIN reviews r
        ON ac.app_id = r.app_id
    GROUP BY ac.category_id, ac.app_id
),

category_concentration AS (
    SELECT
        category_id,
        SUM(app_review_count) AS total_reviews,
        MAX(app_review_count) AS top_app_reviews
    FROM app_review_counts
    GROUP BY category_id
),

category_pricing_base AS (
    SELECT
        ac.category_id,
        pp.price_type,
        pp.price_amount
    FROM apps_categories ac
    JOIN pricing_plans pp
        ON ac.app_id = pp.app_id
),

category_free_entry AS (
    SELECT
        category_id,
        COUNT(*) AS total_plan_count,
        SUM(
            CASE
                WHEN price_type IN ('free', 'free_to_install') THEN 1
                ELSE 0
            END
        ) AS free_entry_plan_count
    FROM category_pricing_base
    GROUP BY category_id
),

category_monthly_prices AS (
    SELECT
        category_id,
        price_amount
    FROM category_pricing_base
    WHERE price_type = 'monthly'
      AND price_amount IS NOT NULL
),

category_price_stats AS (
    SELECT DISTINCT
        category_id,
        COUNT(*) OVER (PARTITION BY category_id) AS monthly_plan_count,
        PERCENTILE_CONT(0.5)
            WITHIN GROUP (ORDER BY price_amount)
            OVER (PARTITION BY category_id) AS median_monthly_price
    FROM category_monthly_prices
)

SELECT
    cas.category,
    cas.app_count,
    cas.developer_count,
    crs.review_count,
    CAST(crs.review_count * 1.0 / cas.app_count AS DECIMAL(10,2)) AS reviews_per_app,
    CAST(cas.avg_rating AS DECIMAL(4,2)) AS avg_rating,
    CAST(crs.one_star_reviews * 100.0 / crs.review_count AS DECIMAL(5,2)) AS one_star_rate_pct,
    CAST(crs.replied_reviews * 100.0 / crs.review_count AS DECIMAL(5,2)) AS reply_rate_pct,
    cps.monthly_plan_count,
    CAST(cps.median_monthly_price AS DECIMAL(10,2)) AS median_monthly_price,
    CAST(
        cfe.free_entry_plan_count * 100.0
        / NULLIF(cfe.total_plan_count, 0)
        AS DECIMAL(5,2)
    ) AS free_entry_share_pct,
    CAST(
        cc.top_app_reviews * 100.0
        / NULLIF(cc.total_reviews, 0)
        AS DECIMAL(5,2)
    ) AS top_app_review_share_pct
FROM category_app_stats cas
JOIN category_review_stats crs
    ON cas.category_id = crs.category_id
LEFT JOIN category_concentration cc
    ON cas.category_id = cc.category_id
LEFT JOIN category_free_entry cfe
    ON cas.category_id = cfe.category_id
LEFT JOIN category_price_stats cps
    ON cas.category_id = cps.category_id
WHERE
    cas.app_count >= 50
    AND crs.review_count >= 1000;
"""

opportunity_df = pd.read_sql(query, conn)

print(opportunity_df.shape)
opportunity_df.head()

C:\Users\xiongsongsong\AppData\Local\Temp\ipykernel_44280\657233545.py:131: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  opportunity_df = pd.read_sql(query, conn)


(616, 12)


,category,app_count,developer_count,review_count,reviews_per_app,avg_rating,one_star_rate_pct,reply_rate_pct,monthly_plan_count,median_monthly_price,free_entry_share_pct,top_app_review_share_pct
0,Baby products,120,115,67122,559.35,3.60,3.25,61.74,115,37.95,42.29,14.66
1,Services,76,72,23699,311.83,4.45,1.56,43.36,155,45.00,25.24,16.47
2,Email automation,53,46,7614,143.66,4.27,1.51,29.85,114,19.00,24.50,19.46
3,Sourcing options - Other,61,59,3522,57.74,2.12,3.72,64.62,37,30.00,48.68,61.10
4,Review highlights,66,63,60228,912.55,3.93,1.23,34.29,107,19.00,28.57,16.42


In [4]:
# Phase 8 - Step 4: Normalize opportunity metrics to 0-100 scores
# Phase 8 - 第四步：将机会评估指标标准化为 0-100 分

score_df = opportunity_df.copy()

def min_max_score(series):
    return (series - series.min()) / (series.max() - series.min()) * 100

# Demand: higher is better
# 需求：越高越好
score_df["demand_review_score"] = min_max_score(
    score_df["review_count"]
)

score_df["demand_reviews_per_app_score"] = min_max_score(
    score_df["reviews_per_app"]
)

# Competition: lower is better
# 竞争：越低越好
score_df["competition_app_score"] = 100 - min_max_score(
    score_df["app_count"]
)

score_df["competition_developer_score"] = 100 - min_max_score(
    score_df["developer_count"]
)

# Customer dissatisfaction: lower rating and higher 1-star rate are better opportunity signals
# 用户不满：评分越低、1-star 占比越高，机会信号越强
score_df["rating_gap_score"] = 100 - min_max_score(
    score_df["avg_rating"]
)

score_df["one_star_score"] = min_max_score(
    score_df["one_star_rate_pct"]
)

# Service gap: lower reply rate is a stronger opportunity signal
# 服务缺口：回复率越低，机会信号越强
score_df["service_gap_score"] = 100 - min_max_score(
    score_df["reply_rate_pct"]
)

# Monetization: higher median monthly price is better
# 变现能力：月费中位数越高越好
score_df["monetization_score"] = min_max_score(
    score_df["median_monthly_price"]
)

# Pricing pressure: lower free-entry share is better
# 免费竞争压力：免费入口占比越低越好
score_df["pricing_pressure_score"] = 100 - min_max_score(
    score_df["free_entry_share_pct"]
)

# Market concentration: lower concentration is better
# 市场集中度：头部 App 占比越低越好
score_df["concentration_score"] = 100 - min_max_score(
    score_df["top_app_review_share_pct"]
)

score_df.head()

,category,app_count,developer_count,review_count,reviews_per_app,avg_rating,one_star_rate_pct,reply_rate_pct,monthly_plan_count,median_monthly_price,...,demand_review_score,demand_reviews_per_app_score,competition_app_score,competition_developer_score,rating_gap_score,one_star_score,service_gap_score,monetization_score,pricing_pressure_score,concentration_score
0,Baby products,120,115,67122,559.35,3.60,3.25,61.74,115,37.95,...,20.287332,51.635896,95.053004,93.185420,40.304183,20.088300,26.179961,22.887299,43.243243,81.450711
1,Services,76,72,23699,311.83,4.45,1.56,43.36,155,45.00,...,6.931710,28.391526,98.162544,96.592710,7.984791,7.652686,51.545680,27.782793,73.361597,78.558875
2,Email automation,53,46,7614,143.66,4.27,1.51,29.85,114,19.00,...,1.984443,12.598839,99.787986,98.652932,14.828897,7.284768,70.190450,9.728491,74.668786,73.781754
3,Sourcing options - Other,61,59,3522,57.74,2.12,3.72,64.62,37,30.00,...,0.725866,4.530173,99.222615,97.622821,96.577947,23.546726,22.205355,17.366850,31.955485,7.253555
4,Review highlights,66,63,60228,912.55,3.93,1.23,34.29,107,19.00,...,18.166943,84.804575,98.869258,97.305864,27.756654,5.224430,64.062931,9.728491,67.479244,78.638760


In [5]:
# Phase 8 - Step 5: Check missing values before scoring
# Phase 8 - 第五步：在计算最终评分前检查缺失值

score_columns = [
    "review_count",
    "reviews_per_app",
    "app_count",
    "developer_count",
    "avg_rating",
    "one_star_rate_pct",
    "reply_rate_pct",
    "median_monthly_price",
    "free_entry_share_pct",
    "top_app_review_share_pct"
]

score_df[score_columns].isna().sum()

review_count                0
reviews_per_app             0
app_count                   0
developer_count             0
avg_rating                  0
one_star_rate_pct           0
reply_rate_pct              0
median_monthly_price        0
free_entry_share_pct        0
top_app_review_share_pct    0
dtype: int64

In [6]:
# Phase 8 - Step 6: Build dimension-level scores
# Phase 8 - 第六步：构建各维度综合评分

score_df["demand_score"] = (
    score_df["demand_review_score"]
    + score_df["demand_reviews_per_app_score"]
) / 2

score_df["competition_score"] = (
    score_df["competition_app_score"]
    + score_df["competition_developer_score"]
) / 2

score_df["customer_gap_score"] = (
    score_df["rating_gap_score"]
    + score_df["one_star_score"]
) / 2

score_df["service_gap_final_score"] = (
    score_df["service_gap_score"]
)

score_df["monetization_final_score"] = (
    score_df["monetization_score"]
)

score_df["pricing_pressure_final_score"] = (
    score_df["pricing_pressure_score"]
)

score_df["concentration_final_score"] = (
    score_df["concentration_score"]
)

score_df[
    [
        "category",
        "demand_score",
        "competition_score",
        "customer_gap_score",
        "service_gap_final_score",
        "monetization_final_score",
        "pricing_pressure_final_score",
        "concentration_final_score"
    ]
].head()

,category,demand_score,competition_score,customer_gap_score,service_gap_final_score,monetization_final_score,pricing_pressure_final_score,concentration_final_score
0,Baby products,35.961614,94.119212,30.196241,26.179961,22.887299,43.243243,81.450711
1,Services,17.661618,97.377627,7.818738,51.545680,27.782793,73.361597,78.558875
2,Email automation,7.291641,99.220459,11.056833,70.190450,9.728491,74.668786,73.781754
3,Sourcing options - Other,2.628019,98.422718,60.062336,22.205355,17.366850,31.955485,7.253555
4,Review highlights,51.485759,98.087561,16.490542,64.062931,9.728491,67.479244,78.638760


In [7]:
# Phase 8 - Step 7: Calculate the final Opportunity Score
# Phase 8 - 第七步：计算最终 Opportunity Score

score_df["opportunity_score"] = (
    score_df["demand_score"] * 0.25
    + score_df["competition_score"] * 0.20
    + score_df["customer_gap_score"] * 0.20
    + score_df["monetization_final_score"] * 0.15
    + score_df["service_gap_final_score"] * 0.10
    + score_df["pricing_pressure_final_score"] * 0.05
    + score_df["concentration_final_score"] * 0.05
)

opportunity_ranking = score_df[
    [
        "category",
        "opportunity_score",
        "demand_score",
        "competition_score",
        "customer_gap_score",
        "monetization_final_score",
        "service_gap_final_score",
        "pricing_pressure_final_score",
        "concentration_final_score"
    ]
].sort_values(
    "opportunity_score",
    ascending=False
)

opportunity_ranking.head(20)

,category,opportunity_score,demand_score,competition_score,customer_gap_score,monetization_final_score,service_gap_final_score,pricing_pressure_final_score,concentration_final_score
428,AI targeting,60.972389,33.475246,99.211891,37.940837,65.280189,91.374551,69.510687,55.360281
285,Automated campaigns,60.019876,34.618253,98.907786,38.541815,54.864246,93.706873,66.242713,59.258668
162,AI optimization,58.328747,28.474439,98.272807,35.021697,58.336227,90.739718,78.201731,56.334878
586,SMS campaigns,58.255674,44.992719,98.158232,28.516271,31.247830,98.233508,67.638226,75.603131
506,ROI analysis,57.455489,21.801512,98.572629,39.221973,65.280189,95.086945,64.564565,38.344784
611,Audience segments,56.700432,19.774950,98.237472,37.038669,65.280189,87.330941,76.011305,47.515578
477,Consent collection,56.689309,43.781868,98.907786,23.493426,31.185334,99.958598,63.593005,68.205784
551,Cost per acquisition,56.523668,30.041990,98.501957,36.248276,37.851538,98.495722,73.167285,57.549129
224,Drip campaigns,56.289849,45.765717,98.158232,20.083684,30.560378,95.749379,66.384031,74.436811
121,Cross-sell emails,55.879978,38.178550,96.742649,28.037978,30.560378,96.798234,66.384031,75.922671


In [8]:
# Phase 8 - Step 8: Inspect the top opportunity candidates
# Phase 8 - 第八步：检查 Opportunity Score 排名前列的候选 Category

top_candidates = score_df.sort_values(
    "opportunity_score",
    ascending=False
).head(20)

top_candidates[
    [
        "category",
        "opportunity_score",
        "app_count",
        "developer_count",
        "review_count",
        "reviews_per_app",
        "avg_rating",
        "one_star_rate_pct",
        "reply_rate_pct",
        "median_monthly_price",
        "free_entry_share_pct",
        "top_app_review_share_pct"
    ]
]

,category,opportunity_score,app_count,developer_count,review_count,reviews_per_app,avg_rating,one_star_rate_pct,reply_rate_pct,median_monthly_price,free_entry_share_pct,top_app_review_share_pct
428,AI targeting,60.972389,51,48,31737,622.29,3.63,5.51,14.50,99.00,27.42,30.99
285,Automated campaigns,60.019876,54,53,34440,637.78,3.61,5.57,12.81,84.00,29.27,28.55
162,AI optimization,58.328747,63,61,32365,513.73,3.77,5.44,14.96,89.00,22.50,30.38
586,SMS campaigns,58.255674,64,63,51403,803.17,3.52,2.38,9.53,49.99,28.48,18.32
506,ROI analysis,57.455489,59,57,23616,400.27,3.82,6.84,11.81,99.00,30.22,41.64
611,Audience segments,56.700432,64,61,22987,359.17,3.96,6.97,17.43,99.00,23.74,35.90
477,Consent collection,56.689309,54,53,43395,803.61,3.85,2.72,8.28,49.90,30.77,22.95
551,Cost per acquisition,56.523668,61,57,33206,544.36,3.69,5.36,9.34,59.50,25.35,29.62
224,Drip campaigns,56.289849,64,63,52274,816.78,4.01,2.62,11.33,49.00,29.19,19.05
121,Cross-sell emails,55.879978,85,80,54947,646.44,3.52,2.25,10.57,49.00,29.19,18.12


In [9]:
# Phase 8 - Step 9: Create a business-valid candidate shortlist
# Phase 8 - 第九步：建立具有商业意义的候选 Category Shortlist

candidate_categories = [
    "AI targeting",
    "Automated campaigns",
    "SMS campaigns",
    "ROI analysis",
    "Audience segments",
    "Consent collection",
    "Browse abandonment",
    "Push notifications"
]

shortlist = score_df[
    score_df["category"].isin(candidate_categories)
].sort_values(
    "opportunity_score",
    ascending=False
)

shortlist[
    [
        "category",
        "opportunity_score",
        "demand_score",
        "competition_score",
        "customer_gap_score",
        "monetization_final_score",
        "service_gap_final_score"
    ]
]

,category,opportunity_score,demand_score,competition_score,customer_gap_score,monetization_final_score,service_gap_final_score
428,AI targeting,60.972389,33.475246,99.211891,37.940837,65.280189,91.374551
285,Automated campaigns,60.019876,34.618253,98.907786,38.541815,54.864246,93.706873
586,SMS campaigns,58.255674,44.992719,98.158232,28.516271,31.247830,98.233508
506,ROI analysis,57.455489,21.801512,98.572629,39.221973,65.280189,95.086945
611,Audience segments,56.700432,19.774950,98.237472,37.038669,65.280189,87.330941
477,Consent collection,56.689309,43.781868,98.907786,23.493426,31.185334,99.958598
32,Browse abandonment,55.751841,36.127544,98.497673,32.643383,30.560378,95.790781
386,Push notifications,55.604953,47.859165,97.294104,18.869696,24.303868,93.348054


In [10]:
# Phase 8 - Step 10: Create the final opportunity shortlist
# Phase 8 - 第十步：建立最终机会候选 Shortlist

final_candidates = [
    "AI targeting",
    "Automated campaigns",
    "SMS campaigns",
    "ROI analysis",
    "Consent collection"
]

final_shortlist = score_df[
    score_df["category"].isin(final_candidates)
].sort_values(
    "opportunity_score",
    ascending=False
)

final_shortlist[
    [
        "category",
        "opportunity_score",
        "app_count",
        "developer_count",
        "review_count",
        "reviews_per_app",
        "avg_rating",
        "one_star_rate_pct",
        "reply_rate_pct",
        "median_monthly_price",
        "free_entry_share_pct",
        "top_app_review_share_pct"
    ]
]

,category,opportunity_score,app_count,developer_count,review_count,reviews_per_app,avg_rating,one_star_rate_pct,reply_rate_pct,median_monthly_price,free_entry_share_pct,top_app_review_share_pct
428,AI targeting,60.972389,51,48,31737,622.29,3.63,5.51,14.50,99.00,27.42,30.99
285,Automated campaigns,60.019876,54,53,34440,637.78,3.61,5.57,12.81,84.00,29.27,28.55
586,SMS campaigns,58.255674,64,63,51403,803.17,3.52,2.38,9.53,49.99,28.48,18.32
506,ROI analysis,57.455489,59,57,23616,400.27,3.82,6.84,11.81,99.00,30.22,41.64
477,Consent collection,56.689309,54,53,43395,803.61,3.85,2.72,8.28,49.90,30.77,22.95


## Final Opportunity Shortlist
## 最终市场机会候选

Based on the multi-factor opportunity scoring model, the three highest-priority market segments are:

根据综合机会评分模型，优先级最高的三个市场方向为：

1. AI Targeting
2. Automated Campaigns
3. SMS Campaigns

ROI Analysis and Consent Collection are retained as secondary opportunity candidates.

ROI Analysis 和 Consent Collection 作为次级候选市场保留。

## Final Recommendations
## 最终推荐

### 1. AI Targeting

**Opportunity / 机会**

AI Targeting ranks first in the opportunity model with an Opportunity Score of 60.97. The category has relatively low competition, with 51 apps and 48 developers, while still generating 31,737 reviews and 622 reviews per app.

AI Targeting 在机会评分模型中排名第一，Opportunity Score 为 60.97。该类别目前只有 51 个 App 和 48 个 Developer，竞争相对较低，但仍累计了 31,737 条 Reviews，平均每个 App 约 622 条 Reviews。

**Evidence / 数据支持**

- Opportunity Score: 60.97
- Apps: 51
- Developers: 48
- Reviews: 31,737
- Reviews per App: 622.29
- Average Rating: 3.63
- 1-Star Review Rate: 5.51%
- Developer Reply Rate: 14.50%

**Risk / 风险**

Demand is meaningful but not as large as the biggest Shopify App categories. AI-related products may also face rapid technological change and increasing future competition.

该类别目前的需求规模虽然可观，但仍低于 Shopify App Marketplace 中最大的成熟市场。同时，AI 产品技术变化较快，未来竞争可能迅速增加.


### 2. Automated Campaigns

**Opportunity / 机会**

Automated Campaigns ranks second with an Opportunity Score of 60.02. The market combines relatively low competition with strong review activity and noticeable customer dissatisfaction.

Automated Campaigns 的 Opportunity Score 为 60.02，排名第二。该市场竞争规模较小，同时拥有较强的用户评论活跃度，并存在一定的用户满意度缺口。

**Evidence / 数据支持**

- Opportunity Score: 60.02
- Apps: 54
- Developers: 53
- Reviews: 34,440
- Reviews per App: 637.78
- Average Rating: 3.61
- 1-Star Review Rate: 5.57%
- Developer Reply Rate: 12.81%

**Risk / 风险**

Marketing automation is already a broad and mature product space. A new entrant would need clear differentiation rather than simply offering another generic campaign automation tool.

Marketing Automation 本身已经是较成熟的产品方向，因此新进入者需要明确的产品差异化，而不能只是提供一个普通的自动化营销工具。


### 3. SMS Campaigns

**Opportunity / 机会**

SMS Campaigns ranks third with an Opportunity Score of 58.26. Among the final three candidates, it shows the strongest review activity, with more than 800 reviews per app, while developer reply rates remain very low.

SMS Campaigns 的 Opportunity Score 为 58.26，排名第三。在最终三个候选中，它的用户评论活跃度最高，平均每个 App 超过 800 条 Reviews，同时 Developer Reply Rate 较低。

**Evidence / 数据支持**

- Opportunity Score: 58.26
- Apps: 64
- Developers: 63
- Reviews: 51,403
- Reviews per App: 803.17
- Average Rating: 3.52
- 1-Star Review Rate: 2.38%
- Developer Reply Rate: 9.53%

**Risk / 风险**

Customer dissatisfaction is less severe than in AI Targeting or Automated Campaigns because the 1-star review rate is relatively low. This means the opportunity may come more from service differentiation and engagement quality than from fixing major product failures.

与 AI Targeting 和 Automated Campaigns 相比，SMS Campaigns 的 1-star Review Rate 较低，因此它的机会可能更多来自服务质量和客户互动方面的差异化，而不是解决严重的产品缺陷。

In [11]:
# Phase 9 - Step 1: Export category metrics for Tableau
# Phase 9 - 第一步：导出 Category 指标供 Tableau 使用

tableau_category_data = score_df[
    [
        "category",
        "app_count",
        "developer_count",
        "review_count",
        "reviews_per_app",
        "avg_rating",
        "one_star_rate_pct",
        "reply_rate_pct",
        "monthly_plan_count",
        "median_monthly_price",
        "free_entry_share_pct",
        "top_app_review_share_pct",
        "demand_score",
        "competition_score",
        "customer_gap_score",
        "service_gap_final_score",
        "monetization_final_score",
        "pricing_pressure_final_score",
        "concentration_final_score",
        "opportunity_score"
    ]
].copy()

tableau_category_data.to_csv(
    r"D:\xiongsongsong\Programming Language\Data Analyst Projects\shopify-app-marketplace-analysis\data\processed\tableau_category_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

print(tableau_category_data.shape)
tableau_category_data.head()

(616, 20)


,category,app_count,developer_count,review_count,reviews_per_app,avg_rating,one_star_rate_pct,reply_rate_pct,monthly_plan_count,median_monthly_price,free_entry_share_pct,top_app_review_share_pct,demand_score,competition_score,customer_gap_score,service_gap_final_score,monetization_final_score,pricing_pressure_final_score,concentration_final_score,opportunity_score
0,Baby products,120,115,67122,559.35,3.60,3.25,61.74,115,37.95,42.29,14.66,35.961614,94.119212,30.196241,26.179961,22.887299,43.243243,81.450711,46.139283
1,Services,76,72,23699,311.83,4.45,1.56,43.36,155,45.00,25.24,16.47,17.661618,97.377627,7.818738,51.545680,27.782793,73.361597,78.558875,42.372688
2,Email automation,53,46,7614,143.66,4.27,1.51,29.85,114,19.00,24.50,19.46,7.291641,99.220459,11.056833,70.190450,9.728491,74.668786,73.781754,39.779214
3,Sourcing options - Other,61,59,3522,57.74,2.12,3.72,64.62,37,30.00,48.68,61.10,2.628019,98.422718,60.062336,22.205355,17.366850,31.955485,7.253555,39.140031
4,Review highlights,66,63,60228,912.55,3.93,1.23,34.29,107,19.00,28.57,16.42,51.485759,98.087561,16.490542,64.062931,9.728491,67.479244,78.638760,50.958527


In [13]:
# Phase 9 - Step 2: Prepare marketplace overview KPIs for Tableau
# Phase 9 - 第二步：准备 Tableau 的 Marketplace Overview KPI 数据

overview_query = """
WITH monthly_price_stats AS (
    SELECT DISTINCT
        PERCENTILE_CONT(0.5)
        WITHIN GROUP (ORDER BY price_amount)
        OVER () AS median_monthly_price
    FROM pricing_plans
    WHERE price_type = 'monthly'
      AND price_amount IS NOT NULL
)

SELECT
    (SELECT COUNT(*) FROM apps) AS total_apps,

    (SELECT COUNT(DISTINCT developer) FROM apps) AS total_developers,

    (SELECT COUNT(*) FROM categories) AS total_categories,

    (SELECT COUNT(*) FROM reviews) AS total_reviews,

    (
        SELECT TOP 1
            CAST(median_monthly_price AS DECIMAL(10,2))
        FROM monthly_price_stats
    ) AS median_monthly_price,

    (
        SELECT CAST(
            SUM(
                CASE
                    WHEN has_developer_reply = 1 THEN 1
                    ELSE 0
                END
            ) * 100.0 / COUNT(*)
            AS DECIMAL(5,2)
        )
        FROM reviews
    ) AS overall_reply_rate_pct;
"""

tableau_overview = pd.read_sql(overview_query, conn)

tableau_overview

C:\Users\xiongsongsong\AppData\Local\Temp\ipykernel_44280\1133440534.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tableau_overview = pd.read_sql(overview_query, conn)


,total_apps,total_developers,total_categories,total_reviews,median_monthly_price,overall_reply_rate_pct
0,11951,7615,1889,1133555,19.99,28.37


In [14]:
# Phase 9 - Step 3: Export marketplace overview KPIs for Tableau
# Phase 9 - 第三步：导出 Marketplace Overview KPI 数据供 Tableau 使用

tableau_overview.to_csv(
    r"D:\xiongsongsong\Programming Language\Data Analyst Projects\shopify-app-marketplace-analysis\data\processed\tableau_marketplace_overview.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Export complete!")

Export complete!


In [16]:
# Phase 9 - Step 4: Recreate complaint theme summary for Tableau
# Phase 9 - 第四步：重新建立投诉主题汇总表供 Tableau 使用

theme_summary_v2 = pd.DataFrame({
    "theme": [
        "Technical issues",
        "Pricing / billing",
        "Customer support",
        "Negative recommendation",
        "Time / effort",
        "Usability"
    ],
    "review_count": [
        9616,
        8761,
        7143,
        2066,
        1397,
        449
    ],
    "percentage": [
        29.19,
        26.60,
        21.69,
        6.27,
        4.24,
        1.36
    ]
})

theme_summary_v2

,theme,review_count,percentage
0,Technical issues,9616,29.19
1,Pricing / billing,8761,26.60
2,Customer support,7143,21.69
3,Negative recommendation,2066,6.27
4,Time / effort,1397,4.24
5,Usability,449,1.36


In [17]:
theme_summary_v2.to_csv(
    r"D:\xiongsongsong\Programming Language\Data Analyst Projects\shopify-app-marketplace-analysis\data\processed\tableau_complaint_themes.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Export complete!")

Export complete!
